In [ ]:
# SNET 

# !pip -q install transformers datasets evaluate scikit-learn pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Libraries loaded for SNET approach.")

Libraries loaded for SNET approach.


In [ ]:
# load dataset and prepare structured source-target input

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def build_snet_input(row):
    src = safe_text(row["text_src"])
    tgt = safe_text(row["text_tgt"])
    return f"Source: {src} [SEP] Edited: {tgt}"

train_df["input_text"] = train_df.apply(build_snet_input, axis=1)
val_df["input_text"] = val_df.apply(build_snet_input, axis=1)
test_df["input_text"] = test_df.apply(build_snet_input, axis=1)

labels = train_df["label"].unique()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nLabel mapping:")
print(label2id)

print("\nSample SNET input:")
print(train_df["input_text"].iloc[0])

print("\nSample label:")
print(train_df["label"].iloc[0], "->", train_df["label_id"].iloc[0])

Train shape: (7478, 10)
Validation shape: (1776, 10)
Test shape: (2312, 10)

Label mapping:
{'Fact/Evidence': 0, 'Grammar': 1, 'Clarity': 2, 'Claim': 3, 'Other': 4}

Sample SNET input:
Source: For MBTI, users were able to provide multiple texts, we report unique users in parentheses. [SEP] Edited: 

Sample label:
Fact/Evidence -> 0


In [ ]:
# convert to Hugging Face datasets

train_snet = Dataset.from_pandas(train_df[["input_text", "label_id"]])
val_snet = Dataset.from_pandas(val_df[["input_text", "label_id"]])
test_snet = Dataset.from_pandas(test_df[["input_text", "label_id"]])

print("Train dataset:", train_snet)
print("Validation dataset:", val_snet)
print("Test dataset:", test_snet)

Train dataset: Dataset({
    features: ['input_text', 'label_id'],
    num_rows: 7478
})
Validation dataset: Dataset({
    features: ['input_text', 'label_id'],
    num_rows: 1776
})
Test dataset: Dataset({
    features: ['input_text', 'label_id'],
    num_rows: 2312
})


In [ ]:
# load tokenizer and model

snet_model_name = "roberta-base"

snet_tokenizer = AutoTokenizer.from_pretrained(snet_model_name)

snet_model = AutoModelForSequenceClassification.from_pretrained(
    snet_model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print("SNET model loaded successfully:", snet_model_name)
print("Number of labels:", len(label2id))
print(type(snet_model))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SNET model loaded successfully: roberta-base
Number of labels: 5
<class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'>


In [ ]:
# tokenize datasets

max_length = 128

def tokenize_snet(example):
    return snet_tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

train_snet_tokenized = train_snet.map(tokenize_snet, batched=True)
val_snet_tokenized = val_snet.map(tokenize_snet, batched=True)
test_snet_tokenized = test_snet.map(tokenize_snet, batched=True)

print("SNET tokenization completed.")
print(train_snet_tokenized[0])

Map:   0%|          | 0/7478 [00:00<?, ? examples/s]

Map:   0%|          | 0/1776 [00:00<?, ? examples/s]

Map:   0%|          | 0/2312 [00:00<?, ? examples/s]

SNET tokenization completed.
{'input_text': 'Source: For MBTI, users were able to provide multiple texts, we report unique users in parentheses. [SEP] Edited: ', 'label_id': 0, 'input_ids': [0, 7061, 35, 286, 17025, 13216, 6, 1434, 58, 441, 7, 694, 1533, 14301, 6, 52, 266, 2216, 1434, 11, 47022, 4, 646, 3388, 510, 742, 34525, 35, 1437, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
# rename labels and set torch format

train_snet_tokenized = train_snet_tokenized.rename_column("label_id", "labels")
val_snet_tokenized = val_snet_tokenized.rename_column("label_id", "labels")
test_snet_tokenized = test_snet_tokenized.rename_column("label_id", "labels")

columns = ["input_ids", "attention_mask", "labels"]

train_snet_tokenized.set_format(type="torch", columns=columns)
val_snet_tokenized.set_format(type="torch", columns=columns)
test_snet_tokenized.set_format(type="torch", columns=columns)

print("SNET dataset ready for training.")
print(train_snet_tokenized[0])

SNET dataset ready for training.
{'labels': tensor(0), 'input_ids': tensor([    0,  7061,    35,   286, 17025, 13216,     6,  1434,    58,   441,
            7,   694,  1533, 14301,     6,    52,   266,  2216,  1434,    11,
        47022,     4,   646,  3388,   510,   742, 34525,    35,  1437,     2,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,    

In [ ]:
# define metrics

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_snet_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("SNET metrics function ready.")

SNET metrics function ready.


In [ ]:
#  train the SNET model

import torch
from transformers import TrainingArguments, Trainer

snet_training_args = TrainingArguments(
    output_dir="./snet_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

snet_trainer = Trainer(
    model=snet_model,
    args=snet_training_args,
    train_dataset=train_snet_tokenized,
    eval_dataset=val_snet_tokenized,
    compute_metrics=compute_snet_metrics
)

snet_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# evaluate on test set

snet_test_results = snet_trainer.evaluate(test_snet_tokenized)
print(snet_test_results)

In [ ]:
# predict on test set

snet_pred_output = snet_trainer.predict(test_snet_tokenized)

print("Prediction step completed.")
print("Prediction shape:", snet_pred_output.predictions.shape)

In [ ]:
# convert predictions to labels

snet_predictions = np.argmax(snet_pred_output.predictions, axis=1)
snet_true_labels = snet_pred_output.label_ids

snet_pred_names = [id2label[i] for i in snet_predictions]
snet_true_names = [id2label[i] for i in snet_true_labels]

for i in range(5):
    print("Pred:", snet_pred_names[i], "| True:", snet_true_names[i])

In [ ]:
# classification report

from sklearn.metrics import classification_report

print(classification_report(snet_true_names, snet_pred_names, labels=list(label2id.keys()), zero_division=0))

In [ ]:
# confusion matrix

from sklearn.metrics import confusion_matrix

snet_label_list = list(label2id.keys())

snet_cm = confusion_matrix(snet_true_names, snet_pred_names, labels=snet_label_list)

import pandas as pd
snet_cm_df = pd.DataFrame(snet_cm, index=snet_label_list, columns=snet_label_list)
print(snet_cm_df)

plt.figure(figsize=(8, 6))
plt.imshow(snet_cm, interpolation="nearest")
plt.title("SNET Confusion Matrix")
plt.colorbar()
plt.xticks(range(len(snet_label_list)), snet_label_list, rotation=45)
plt.yticks(range(len(snet_label_list)), snet_label_list)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
#  save model to Google Drive

from google.colab import drive
drive.mount('/content/drive')

snet_save_path = "/content/drive/MyDrive/snet_final_model"

snet_trainer.save_model(snet_save_path)
snet_tokenizer.save_pretrained(snet_save_path)

print("SNET model saved at:", snet_save_path)